# Pilot Generation — LLM-side Dataset 2

Generates a small pilot batch across all 5 Core prompt families using one seen generator
(OpenAI `gpt-4o-mini`) to verify prompt quality, QC pipeline, and output format **before** a full run.

Per `v2/docs/llm_prompt_families_contract.md` Generation Workflow — Step 2:
- Coverage: all subtypes in every family
- Counts: 5 examples per subtype (lower than the 10–15 recommended for full pilot, for fast smoke-test)
- Generator: `openai/gpt-4o-mini` (Seen A)
- Output: `v2/data/interim/llm-tests/pilot_<run_stem>.jsonl` + `pilot_qc_<run_stem>.json` (unique **run stem** every time you execute the setup cell — prior pilot files are never overwritten; re-run cell 1 before each new pilot to refresh the stem)

**QC axes applied (§ Acceptance and Retry Protocol + Step 3):**
1. Format — no markdown, no headers, no wrapper phrases
2. Masking — no raw URLs, emails, or phone numbers
3. Length — token count assigned to actual `length_bin` via `v2/config.py`; out-of-family bins trigger retry
4. Deduplication — near-duplicate detection within pilot (SequenceMatcher ≥ 0.85)

**Does NOT inject `few_shot_anchors` at runtime** — only `system_prompt` + `user_template`.

Length-bin sampling for `legitimate_email` in this pilot: **35% short / 50% medium / 15% long** (pilot override; mass generation uses the contract table: 65% / 25% / 10%). Other families match `llm_prompt_families_contract.md`.

In [49]:
# ── Cell 1: Setup & dependency check ─────────────────────────────────────
from __future__ import annotations

import json
import os
import re
import sys
import uuid
import random
import datetime
from difflib import SequenceMatcher
from pathlib import Path

# ── paths ──
BASE        = Path("__file__").resolve().parent.parent.parent   # v2/
BASE        = Path("/Users/askar/projects/antifraud-deepfake-detection/v2")
PROMPTS_DIR = BASE / "data" / "prompts"
OUTPUT_DIR  = BASE / "data" / "interim" / "llm-tests"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Unique filename stem for this pilot run. Set in cell 1; reused by the save cell.
# Microseconds + short uuid avoid collisions if you re-run twice in the same second.
PILOT_RUN_STEM = (
    datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S_%f")
    + "_"
    + uuid.uuid4().hex[:8]
)
print("PILOT_RUN_STEM (use for this session):", PILOT_RUN_STEM)

# ── v2/config.py for length_bin ──
sys.path.insert(0, str(BASE))
from config import length_bin as compute_length_bin

# ── third-party ──
try:
    from dotenv import load_dotenv
    from openai import OpenAI
    from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
    import pandas as pd
except ModuleNotFoundError as e:
    raise SystemExit(
        f"Missing dependency: {e}. "
        "Activate the project venv (uv sync) and re-run this kernel."
    )

# ── API key ──
load_dotenv(BASE.parent / ".env")
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise SystemExit(
        "OPENAI_API_KEY not found. "
        "Add it to /Users/askar/projects/antifraud-deepfake-detection/.env"
    )

client = OpenAI(api_key=api_key)

# ── generation config ──
MODEL       = "gpt-4o-mini"           # Seen generator A
TEMPERATURE = 0.9                     # API parameter only
MAX_TOKENS  = 900
MAX_RETRIES = 3
SAMPLES_PER_SUBTYPE = 5               # pilot: 5 per subtype; increase to 10-15 for full pilot

# ── length bin split per family (contract + pilot override for legitimate_email) ──
BIN_SPLITS = {
    "phishing_email":          ["short"]*20 + ["medium"]*50 + ["long"]*30,
    "advance_fee_scam_email":  ["medium"]*40 + ["long"]*60,
    "fraud_sms_deceptive":     ["short"]*15 + ["medium"]*85,
    "legitimate_email":        ["short"]*35 + ["medium"]*50 + ["long"]*15,  # pilot: 35/50/15; contract 65/25/10
    "legitimate_sms":          ["short"]*100,
}

print(f"BASE:        {BASE}")
print(f"PROMPTS_DIR: {PROMPTS_DIR}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")
print(f"MODEL:       {MODEL}")
print(f"Samples/subtype: {SAMPLES_PER_SUBTYPE}")

# ── verify prompt files exist ──
FAMILIES = [
    "phishing_email",
    "advance_fee_scam_email",
    "fraud_sms_deceptive",
    "legitimate_email",
    "legitimate_sms",
]
specs = {}
for name in FAMILIES:
    path = PROMPTS_DIR / f"{name}.json"
    assert path.exists(), f"Missing prompt spec: {path}"
    specs[name] = json.loads(path.read_text())
    print(f"  loaded {name}.json v{specs[name]['version']} — {len(specs[name]['subtypes'])} subtypes")

print("\nAll specs loaded. Dependencies OK.")

PILOT_RUN_STEM (use for this session): 20260408_005559_455333_55017f6f
BASE:        /Users/askar/projects/antifraud-deepfake-detection/v2
PROMPTS_DIR: /Users/askar/projects/antifraud-deepfake-detection/v2/data/prompts
OUTPUT_DIR:  /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-tests
MODEL:       gpt-4o-mini
Samples/subtype: 5
  loaded phishing_email.json v7.0 — 9 subtypes
  loaded advance_fee_scam_email.json v6.0 — 4 subtypes
  loaded fraud_sms_deceptive.json v2.0 — 3 subtypes
  loaded legitimate_email.json v7.0 — 5 subtypes
  loaded legitimate_sms.json v2.0 — 2 subtypes

All specs loaded. Dependencies OK.


In [50]:
# ── Cell 2: QC helpers ───────────────────────────────────────────────────

# ── format QC ──
WRAPPER_PREFIXES = (
    "here is", "here's", "certainly", "sure!", "sure,", "as requested",
    "of course", "absolutely", "below is", "the following",
)

# SMS: no emojis / pictographs (matches v2/docs/llm_prompt_families_contract.md)
_SMS_EMOJI_RE = re.compile(
    r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F600-\U0001F64F\U0001F1E0-\U0001F1FF]+",
    flags=re.UNICODE,
)


def format_qc(text: str, channel: str | None = None) -> list[str]:
    issues = []
    if not text or len(text.split()) < 3:
        issues.append("empty_or_near_empty")
    if re.search(r"^#{1,6}\s", text, re.MULTILINE):
        issues.append("markdown_header")
    if re.search(r"\*\*|__|\*[^*]|_[^_]", text):
        issues.append("markdown_emphasis")
    if re.search(r"<[a-zA-Z][^>]*>", text):
        issues.append("html_tag")
    if any(text.lower().lstrip().startswith(w) for w in WRAPPER_PREFIXES):
        issues.append("wrapper_phrase")
    if channel == "sms" and _SMS_EMOJI_RE.search(text):
        issues.append("emoji_in_sms")
    return issues

# ── masking QC ──
def masking_qc(text: str) -> list[str]:
    issues = []
    if re.search(r"https?://", text):
        issues.append("unmasked_url")
    if re.search(r"\b[\w.+-]+@[\w-]+\.\w+", text):
        issues.append("unmasked_email")
    if re.search(r"\b(\+?\d[\d\s\-().]{6,}\d)\b", text):
        issues.append("unmasked_phone")
    return issues

# ── length QC ──
def length_qc(text: str, channel: str, family_bins: list[str]) -> dict:
    token_count = len(text.split())
    actual_bin  = compute_length_bin(token_count, channel)
    accepted    = actual_bin in family_bins
    return {
        "token_count": token_count,
        "actual_bin":  actual_bin,
        "accepted":    accepted,
    }

# ── combined validation ──
def validate(text: str, channel: str, family_bins: list[str]) -> dict:
    fmt   = format_qc(text, channel=channel)
    mask  = masking_qc(text)
    lqc   = length_qc(text, channel, family_bins)
    issues = fmt + mask + ([] if lqc["accepted"] else [f"bin_out_of_family:{lqc['actual_bin']}"])
    return {
        "passed":      len(issues) == 0,
        "issues":      issues,
        "token_count": lqc["token_count"],
        "actual_bin":  lqc["actual_bin"],
    }

# ── near-dedup ──
def is_near_duplicate(text: str, seen_texts: list[str], threshold: float = 0.85) -> bool:
    for s in seen_texts:
        if SequenceMatcher(None, text, s).ratio() > threshold:
            return True
    return False

print("QC helpers defined.")

QC helpers defined.


In [51]:
# ── Cell 3: Generation function with retry logic ─────────────────────────
from tenacity import (
    retry, stop_after_attempt, wait_exponential, retry_if_exception_type
)
from openai import RateLimitError, APIStatusError

@retry(
    retry=retry_if_exception_type((RateLimitError, APIStatusError)),
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1, min=2, max=30),
)
def _api_call(system_msg: str, user_msg: str) -> str:
    """Single API call; raises on error (tenacity handles retries)."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg},
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content.strip()


def generate_one(
    spec: dict,
    subtype: str,
    target_bin: str,
    seen_texts: list[str],
) -> dict | None:
    """
    Generate one text for the given family/subtype/bin.
    Returns a record dict on success, or None if all retries fail.
    """
    family     = spec["scenario_family"]
    channel    = spec["channel"]
    family_bins = spec["length_bins"]
    length_hint = spec["length_bin_word_guide"][target_bin]

    system_msg = spec["system_prompt"]
    user_msg   = spec["user_template"].format(
        subtype=subtype,
        length_bin=target_bin,
        length_hint=length_hint,
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            text = _api_call(system_msg, user_msg)
        except Exception as exc:
            print(f"    [API error] {family}/{subtype}: {exc}")
            return None

        v = validate(text, channel, family_bins)

        if not v["passed"]:
            print(f"    [QC fail {attempt}/{MAX_RETRIES}] {family}/{subtype}: {v['issues']}")
            continue  # retry

        if is_near_duplicate(text, seen_texts):
            print(f"    [near-dup {attempt}/{MAX_RETRIES}] {family}/{subtype}: regenerating")
            continue

        # ── success ──
        seen_texts.append(text)
        return {
            "gen_id":          str(uuid.uuid4()),
            "text":            text,
            "label":           1,
            "label_str":       "llm",
            "fraudness":       spec["fraudness"],
            "channel":         channel,
            "scenario_family": family,
            "subtype":         subtype,
            "target_bin":      target_bin,
            "actual_bin":      v["actual_bin"],
            "token_count":     v["token_count"],
            "origin_model":    f"openai/{MODEL}",
            "split":           "pilot",
            "generated_at":    datetime.datetime.utcnow().isoformat(),
            "qc_issues":       [],
        }

    # all retries exhausted
    print(f"    [SKIP] {family}/{subtype}/{target_bin}: all {MAX_RETRIES} retries failed")
    return None


print("Generation function ready.")

Generation function ready.


In [52]:
# ── Cell 4: Run pilot generation ─────────────────────────────────────────
#
# For each family → each subtype → SAMPLES_PER_SUBTYPE calls.
# Length bin for each call is sampled from BIN_SPLITS (see cell 1; contract: v2/docs/llm_prompt_families_contract.md).
# All output collected into `records` list; skipped calls logged separately.

random.seed(42)

records  : list[dict] = []
skipped  : list[dict] = []
# per-family seen texts for dedup (not cross-family)
seen_by_family: dict[str, list[str]] = {f: [] for f in FAMILIES}

for family_name, spec in specs.items():
    subtypes = spec["subtypes"]
    bins_pool = BIN_SPLITS[family_name]
    print(f"\n{'='*60}")
    print(f"Family: {family_name}  ({len(subtypes)} subtypes × {SAMPLES_PER_SUBTYPE} samples)")
    print(f"{'='*60}")

    for subtype in subtypes:
        print(f"  subtype: {subtype}")
        for i in range(SAMPLES_PER_SUBTYPE):
            target_bin = random.choice(bins_pool)
            record = generate_one(
                spec=spec,
                subtype=subtype,
                target_bin=target_bin,
                seen_texts=seen_by_family[family_name],
            )
            if record is not None:
                records.append(record)
                print(f"    [{i+1}/{SAMPLES_PER_SUBTYPE}] OK  "
                      f"target={target_bin} actual={record['actual_bin']} "
                      f"tokens={record['token_count']}")
            else:
                skipped.append({"family": family_name, "subtype": subtype,
                                "target_bin": target_bin, "attempt": i+1})

print(f"\n{'='*60}")
print(f"Pilot complete: {len(records)} generated, {len(skipped)} skipped")
print(f"{'='*60}")


Family: phishing_email  (9 subtypes × 5 samples)
  subtype: account_suspension
    [1/5] OK  target=long actual=medium tokens=288
    [2/5] OK  target=short actual=short tokens=37
    [3/5] OK  target=short actual=short tokens=36
    [4/5] OK  target=long actual=medium tokens=296
    [5/5] OK  target=medium actual=short tokens=72
  subtype: suspicious_activity_or_login
    [1/5] OK  target=medium actual=short tokens=68
    [2/5] OK  target=medium actual=short tokens=87
    [3/5] OK  target=short actual=short tokens=35
    [4/5] OK  target=long actual=medium tokens=293
    [5/5] OK  target=short actual=short tokens=33
  subtype: invoice_or_payment_lure
    [1/5] OK  target=long actual=medium tokens=282
    [2/5] OK  target=long actual=medium tokens=246
    [3/5] OK  target=medium actual=short tokens=78
    [4/5] OK  target=short actual=short tokens=33
    [QC fail 1/3] phishing_email/invoice_or_payment_lure: ['unmasked_phone']
    [5/5] OK  target=long actual=medium tokens=261
  subtyp

In [53]:
# ── Cell 5: Save outputs ──────────────────────────────────────────────────
# Uses PILOT_RUN_STEM from cell 1 so repeated full-notebook runs do not overwrite
# previous pilots. Re-execute cell 1 first if you want a new run stem.

out_jsonl  = OUTPUT_DIR / f"pilot_{PILOT_RUN_STEM}.jsonl"
out_qc     = OUTPUT_DIR / f"pilot_qc_{PILOT_RUN_STEM}.json"

# ── write JSONL ──
with open(out_jsonl, "w") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# ── write QC summary ──
df = pd.DataFrame(records)

qc_summary = {
    "model":              MODEL,
    "generated_at":       PILOT_RUN_STEM,
    "total_generated":    len(records),
    "total_skipped":      len(skipped),
    "skipped_log":        skipped,
    "by_family": {},
}

for family_name in FAMILIES:
    sub = df[df["scenario_family"] == family_name] if len(df) > 0 else pd.DataFrame()
    if len(sub) == 0:
        qc_summary["by_family"][family_name] = {"count": 0}
        continue
    qc_summary["by_family"][family_name] = {
        "count":            len(sub),
        "by_subtype":       sub["subtype"].value_counts().to_dict(),
        "by_actual_bin":    sub["actual_bin"].value_counts().to_dict(),
        "by_target_bin":    sub["target_bin"].value_counts().to_dict(),
        "bin_match_rate":   float((sub["actual_bin"] == sub["target_bin"]).mean()),
        "token_p50":        float(sub["token_count"].median()),
        "token_p90":        float(sub["token_count"].quantile(0.9)),
    }

with open(out_qc, "w") as f:
    json.dump(qc_summary, f, ensure_ascii=False, indent=2)

print(f"Saved JSONL:   {out_jsonl}  ({len(records)} rows)")
print(f"Saved QC log:  {out_qc}")

Saved JSONL:   /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-tests/pilot_20260408_005559_455333_55017f6f.jsonl  (115 rows)
Saved QC log:  /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-tests/pilot_qc_20260408_005559_455333_55017f6f.json


In [54]:
# ── Cell 6: Pilot QC summary ─────────────────────────────────────────────

if len(records) == 0:
    print("No records generated — check API key and model access.")
else:
    print("=" * 65)
    print(f"PILOT QC SUMMARY  —  model: {MODEL}")
    print("=" * 65)

    for family_name in FAMILIES:
        fam = df[df["scenario_family"] == family_name]
        if len(fam) == 0:
            print(f"\n{family_name}: 0 records")
            continue

        print(f"\n{family_name}  ({len(fam)} records)")
        print(f"  subtypes covered:  {sorted(fam['subtype'].unique())}")
        print(f"  actual_bin dist:   {fam['actual_bin'].value_counts().to_dict()}")
        print(f"  target_bin dist:   {fam['target_bin'].value_counts().to_dict()}")
        bin_match = (fam["actual_bin"] == fam["target_bin"]).mean()
        print(f"  bin match rate:    {bin_match:.0%}")
        print(f"  token p50/p90:     {fam['token_count'].median():.0f} / "
              f"{fam['token_count'].quantile(0.9):.0f}")

    print(f"\nTotal: {len(records)} generated, {len(skipped)} skipped")
    if skipped:
        print("Skipped:")
        for s in skipped:
            print(f"  {s}")

PILOT QC SUMMARY  —  model: gpt-4o-mini

phishing_email  (45 records)
  subtypes covered:  ['account_suspension', 'account_verification', 'card_verification_or_card_issue', 'invoice_or_payment_lure', 'kyc_or_identity_update', 'nonfinancial_phishing', 'password_reset', 'refund_or_reward_lure', 'suspicious_activity_or_login']
  actual_bin dist:   {'short': 29, 'medium': 16}
  target_bin dist:   {'medium': 19, 'long': 15, 'short': 11}
  bin match rate:    27%
  token p50/p90:     85 / 295

advance_fee_scam_email  (20 records)
  subtypes covered:  ['business_investment_opportunity', 'diplomatic_package_or_fund_release', 'inheritance_or_estate_transfer', 'oil_contract_or_large_transfer']
  actual_bin dist:   {'medium': 16, 'long': 4}
  target_bin dist:   {'long': 12, 'medium': 8}
  bin match rate:    60%
  token p50/p90:     371 / 422

fraud_sms_deceptive  (15 records)
  subtypes covered:  ['account_alert', 'delivery_fee_or_service_issue', 'prize_or_contest_scam']
  actual_bin dist:   {'med

In [55]:
# ── Cell 7: Manual semantic QC sample ────────────────────────────────────
#
# Print 1 random sample per subtype for human review.
# Use this cell to check:
#   - subtype reads correctly
#   - family didn't drift to adjacent genre
#   - masking applied (no raw URLs/emails/phones)
#   - no wrapper phrases or markdown
#   - appropriate length and tone

if len(records) == 0:
    print("No records to review.")
else:
    random.seed(99)
    reviewed_subtypes = set()

    for family_name in FAMILIES:
        fam = df[df["scenario_family"] == family_name]
        if len(fam) == 0:
            continue
        print(f"\n{'━'*65}")
        print(f" {family_name.upper()}")
        print(f"{'━'*65}")

        for subtype in sorted(fam["subtype"].unique()):
            sub_rows = fam[fam["subtype"] == subtype]
            row = sub_rows.sample(1).iloc[0]
            print(f"\n  [{subtype}]  bin={row['actual_bin']}  tokens={row['token_count']}")
            print(f"  {'-'*60}")
            # wrap text for readability
            import textwrap
            for line in textwrap.wrap(row["text"], width=60, break_long_words=False):
                print(f"  {line}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 PHISHING_EMAIL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [account_suspension]  bin=short  tokens=37
  ------------------------------------------------------------
  IMPORTANT ACCOUNT NOTICE   your account is about to be
  suspended due to suspicious activity. you must verify your
  identity immediately to prevent loss of access. act fast and
  confirm here [URL] or your account will be locked.

  [account_verification]  bin=short  tokens=87
  ------------------------------------------------------------
  IMMEDIATE ACTION REQUIRED   Your account verification is
  incomplete. This is very serious. You must confirm your
  information to avoid suspension.   Failure to act now will
  result in your account being locked in 24 hours. We cannot
  guarantee access after this time! Click here to verify your
  details: [URL].  If you do not update your account
  verification details, you will lose all a

In [56]:
# ── Cell 8: Template repetition check ────────────────────────────────────
#
# Checks opening phrases for repetition within each family × subtype slice.
# Red flag: >30% identical openings in any slice.

if len(records) == 0:
    print("No records to check.")
else:
    print("Opening phrase repetition check (first 8 words):")
    print()

    from collections import Counter

    for family_name in FAMILIES:
        fam = df[df["scenario_family"] == family_name]
        if len(fam) < 2:
            continue

        for subtype in sorted(fam["subtype"].unique()):
            sub_rows = fam[fam["subtype"] == subtype]
            if len(sub_rows) < 2:
                continue

            openings = [
                " ".join(t["text"].split()[:8])
                for _, t in sub_rows.iterrows()
            ]
            most_common, count = Counter(openings).most_common(1)[0]
            pct = count / len(openings)

            flag = " ⚠ HIGH" if pct > 0.3 else ""
            print(f"  {family_name}/{subtype}: "
                  f"top opening = {pct:.0%} — \"{most_common}...\"{flag}")

Opening phrase repetition check (first 8 words):

  phishing_email/account_suspension: top opening = 20% — "Your Account is at Risk We regret to..."
  phishing_email/account_verification: top opening = 20% — "URGENT ACCOUNT ALERT Your account verification process is..."
  phishing_email/card_verification_or_card_issue: top opening = 20% — "URGENT: CARD VERIFICATION REQUIRED your card has been..."
  phishing_email/invoice_or_payment_lure: top opening = 20% — "This is your final notification regarding unpaid account..."
  phishing_email/kyc_or_identity_update: top opening = 20% — "URGENT ACTION REQUIRED Your identity verification is incomplete...."
  phishing_email/nonfinancial_phishing: top opening = 20% — "URGENT ACCOUNT NOTICE we detected abnormal activity on..."
  phishing_email/password_reset: top opening = 20% — "ACTION REQUIRED IMMEDIATELY Your password reset request is..."
  phishing_email/refund_or_reward_lure: top opening = 20% — "Urgent Refund Notice We regret to inform you...